# EEG Power Classifier: Inference

Make predictions using frontal theta & posterior alpha features.

- **Frontal Theta**: 4-8 Hz at Fz/FCz (focused attention marker)
- **Posterior Alpha**: 8-12 Hz at Pz/O1/O2 (reduced sensory processing)

## 1. Setup

In [ ]:
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

env_path = Path('.').resolve() / '.env'
load_dotenv(env_path)

from eeg_processor import Config, DataLoader, EEGPowerExtractor, HRVClassifier

print(f"Setup complete")

## 2. Load Model

In [ ]:
config = Config(bids_root=os.getenv('BIDS_ROOT'), random_seed=42)

model_dir = Path('.').resolve() / 'models' / 'eeg_power_classifier'

if not model_dir.exists():
    print(f"[ERROR] Model not found at {model_dir}")
    print("Train using train_eeg_power_classifier.ipynb first")
else:
    classifier = HRVClassifier(input_size=2)
    classifier.load(str(model_dir))
    print(f"[OK] Model loaded")

## 3. Load Data

In [ ]:
loader = DataLoader(config)
loader.explore_bids_structure()

MEDITATE_TASK = 'med1breath'
THINK_TASK = 'think1'

# Load data for first 3 subjects
loaded_data = {}
for subject_id in loader.subjects[:3]:
    raw_med = loader.load_eeg_data(subject_id, session=None, task=MEDITATE_TASK)
    raw_think = loader.load_eeg_data(subject_id, session=None, task=THINK_TASK)
    
    if raw_med and raw_think:
        loaded_data[subject_id] = {MEDITATE_TASK: raw_med, THINK_TASK: raw_think}

print(f"Loaded {len(loaded_data)} subjects")

## 4. Extract & Predict

In [ ]:
extractor = EEGPowerExtractor(sampling_rate=500)
channel_mapping = {'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog',
                   'EXG5': 'misc', 'EXG6': 'misc', 'EXG7': 'ecg'}

results = []

for subject_id, tasks_data in loaded_data.items():
    print(f"\nSubject {subject_id}:")
    
    for task_name, raw in tasks_data.items():
        # Setup channels
        loader.raw = raw
        loader.setup_montage(montage_name='biosemi64', on_missing='ignore')
        loader.set_channel_types(channel_mapping)
        
        # Extract features
        try:
            features, feat_array = extractor.extract_from_raw(raw)
            
            # Predict
            X = feat_array.reshape(1, -1)
            pred_class, pred_probs = classifier.predict_with_proba(X)
            
            class_name = "Think" if pred_class[0] == 0 else "Meditate"
            true_name = "Meditate" if task_name == MEDITATE_TASK else "Think"
            
            print(f"  {task_name}:")
            print(f"    Theta: {features['frontal_theta']:.2f} µV², Alpha: {features['posterior_alpha']:.2f} µV²")
            print(f"    True: {true_name}, Pred: {class_name}")
            print(f"    Probabilities: Think={pred_probs[0][0]:.3f}, Meditate={pred_probs[0][1]:.3f}")
            
            results.append({
                'Subject': subject_id,
                'Task': task_name,
                'Theta': features['frontal_theta'],
                'Alpha': features['posterior_alpha'],
                'Predicted': class_name,
                'Think_Prob': pred_probs[0][0],
                'Med_Prob': pred_probs[0][1]
            })
        except Exception as e:
            print(f"  Error: {e}")

## 5. Results Summary

In [ ]:
if results:
    df = pd.DataFrame(results)
    print("\nResults:")
    print(df.to_string(index=False))

## 6. Visualization

In [ ]:
if results:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Theta vs Alpha
    meditate_data = [r for r in results if 'med1' in r['Task']]
    think_data = [r for r in results if 'think' in r['Task']]
    
    if meditate_data:
        theta_med = [r['Theta'] for r in meditate_data]
        alpha_med = [r['Alpha'] for r in meditate_data]
        axes[0].scatter(theta_med, alpha_med, label='Meditate', s=100, alpha=0.7, color='blue')
    
    if think_data:
        theta_think = [r['Theta'] for r in think_data]
        alpha_think = [r['Alpha'] for r in think_data]
        axes[0].scatter(theta_think, alpha_think, label='Think', s=100, alpha=0.7, color='red')
    
    axes[0].set_xlabel('Frontal Theta (µV²)')
    axes[0].set_ylabel('Posterior Alpha (µV²)')
    axes[0].set_title('Feature Space')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Probabilities
    tasks = [r['Task'] for r in results]
    think_probs = [r['Think_Prob'] for r in results]
    med_probs = [r['Med_Prob'] for r in results]
    
    x = np.arange(len(results))
    width = 0.35
    
    axes[1].bar(x - width/2, think_probs, width, label='Think', alpha=0.8)
    axes[1].bar(x + width/2, med_probs, width, label='Meditate', alpha=0.8)
    axes[1].set_ylabel('Probability')
    axes[1].set_title('Prediction Probabilities')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f"{r['Subject']}_{r['Task'][:4]}" for r in results], rotation=45)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()